<div style="background: linear-gradient(135deg, #1d4ed8, #3b82f6); padding: 2rem; border-radius: 16px; color: white; text-align: center;">
  <h1 style="font-size: 2.5rem; margin: 0;">🚗 Drive Wise</h1>
  <p style="font-size: 1.2rem; margin: 0.5rem 0 0 0; opacity: 0.9;">Metadata-Aware Automotive RAG Assistant</p>
  <p style="font-size: 0.9rem; margin: 0.5rem 0 0 0; opacity: 0.75;">Powered by Google Gemini 2.5 Flash · Hybrid Semantic + Keyword Retrieval</p>
</div>

---

## 📋 What This Notebook Does

This is a **complete, self-contained demo** of the DriveWise RAG pipeline. It:

1. **Installs** all dependencies automatically
2. **Downloads** the pre-built brochure index (10 cars, ~13MB) from GitHub
3. **Demonstrates** the full RAG pipeline — retrieval → grounded generation → citations
4. **Evaluates** answer quality using LLM-as-a-Judge
5. **Lets you ask your own questions** interactively

### 🚗 Cars Available in the Index
| Brand | Model |
|-------|-------|
| Honda | Amaze |
| Hyundai | Aura |
| Hyundai | Grand I10 Nios |
| Hyundai | Verna |
| Mahindra | Scorpio N |
| Mahindra | Thar |
| Mahindra | XUV700 |
| Tata | Sierra |
| Toyota | Fortuner |
| Toyota | Innova Hycross |

> **⚡ Quick Start**: Run all cells top-to-bottom. The only thing you need is a **Google Gemini API key** (free at [aistudio.google.com](https://aistudio.google.com)).

## Step 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q google-generativeai pypdf numpy
print("✅ All packages installed!")

## Step 2 — Configure Your Gemini API Key

Get a **free** API key from [aistudio.google.com/apikey](https://aistudio.google.com/apikey)

**On Google Colab**: Use the 🔑 Secrets panel (left sidebar) → add `GOOGLE_API_KEY`  
**Locally**: Set the environment variable before running

In [ ]:
import os
import json
import time
import numpy as np
import google.generativeai as genai

# ── API Key Configuration ──────────────────────────────────────────────────────
api_key = None

# Try Colab Secrets first (recommended — no hardcoding!)
try:
    from google.colab import userdata
    api_key = userdata.get('GOOGLE_API_KEY')
    print("🔑 API key loaded from Colab Secrets.")
except Exception:
    pass

# Fallback: environment variable
if not api_key:
    api_key = os.environ.get('GOOGLE_API_KEY')
    if api_key:
        print("🔑 API key loaded from environment variable.")

# Last resort: paste it directly (replace the string below)
if not api_key:
    api_key = "YOUR_GEMINI_API_KEY_HERE"  # ← Replace this if needed
    print("⚠️  Using hardcoded API key — consider using Colab Secrets instead.")

genai.configure(api_key=api_key)
print("✅ Gemini API configured!")

## Step 3 — Download the Pre-Built Brochure Index

The index contains pre-computed embeddings for all 10 car brochures (~13MB). Downloading it saves you from running the expensive indexing step.

In [ ]:
import urllib.request

# ── Download the pre-built index from GitHub ───────────────────────────────────
INDEX_URL = "https://raw.githubusercontent.com/avanishar/drivewiseapp/main/index/brochure_index.json"
INDEX_PATH = "brochure_index.json"

if not os.path.exists(INDEX_PATH):
    print("⬇️  Downloading brochure index (~13MB)... this takes ~30 seconds.")
    try:
        urllib.request.urlretrieve(INDEX_URL, INDEX_PATH)
        print(f"✅ Index downloaded successfully! ({os.path.getsize(INDEX_PATH)/1024/1024:.1f} MB)")
    except Exception as e:
        print(f"❌ Download failed: {e}")
        print("\n📌 Manual option: Upload brochure_index.json from the project's index/ folder to this notebook.")
else:
    print(f"✅ Index already present ({os.path.getsize(INDEX_PATH)/1024/1024:.1f} MB) — skipping download.")

# ── Load index into memory ─────────────────────────────────────────────────────
print("\n⏳ Loading index into memory...")
with open(INDEX_PATH, 'r', encoding='utf-8') as f:
    index_data = json.load(f)

# Convert embeddings from list → numpy arrays for fast vector math
for chunk in index_data.get('chunks', []):
    if 'embedding' in chunk and chunk['embedding'] is not None:
        chunk['embedding'] = np.array(chunk['embedding'], dtype=np.float32)

# Summary
files = index_data.get('files', {})
chunks = index_data.get('chunks', [])
print(f"✅ Index loaded: {len(chunks)} chunks from {len(files)} car brochures")
print("\n📁 Indexed Cars:")
print("-" * 50)
for fname, meta in files.items():
    print(f"  🚗 {meta.get('brand')} {meta.get('model'):25s} → {meta.get('chunks_count')} chunks")

## Step 4 — The RAG Engine

This implements the full retrieval pipeline:
- **Metadata Pre-Filter** → only chunks for the selected car
- **Semantic Search** → cosine similarity on Gemini embeddings
- **Keyword Boost** → exact match scoring for spec numbers
- **Hybrid Ranking** → 80% semantic + 20% keyword

In [ ]:
# ── Stopwords for keyword scoring ──────────────────────────────────────────────
STOPWORDS = {
    "a","about","above","after","again","against","all","am","an","and","any","are",
    "as","at","be","because","been","before","being","below","between","both","but",
    "by","can","did","do","does","doing","down","during","each","for","from","had",
    "has","have","having","he","her","here","him","his","how","i","if","in","into",
    "is","it","its","me","more","most","my","no","nor","not","of","off","on","once",
    "only","or","other","our","out","over","own","same","she","should","so","some",
    "such","than","that","the","their","them","then","there","these","they","this",
    "those","through","to","too","under","until","up","very","was","we","were","what",
    "when","where","which","while","who","whom","why","with","you","your"
}


def cosine_similarity(v1, v2):
    """Computes cosine similarity between two embedding vectors."""
    dot = np.dot(v1, v2)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    return float(dot / (n1 * n2)) if n1 > 0 and n2 > 0 else 0.0


def keyword_score(query, text):
    """Exact keyword overlap score — useful for numbers and spec names."""
    words = [w.strip("?,.:;!\"'()").lower() for w in query.split()]
    words = [w for w in words if w and w not in STOPWORDS]
    if not words:
        return 0.0
    text_lower = text.lower()
    return sum(1 for w in words if w in text_lower) / len(words)


def retrieve_chunks(query, brand, model, limit=4, target_section=None):
    """
    Retrieves the top-K most relevant brochure chunks for a query.
    Uses metadata pre-filtering + hybrid semantic/keyword scoring.
    """
    all_chunks = index_data.get('chunks', [])

    # 1. Metadata Pre-Filter: only chunks for selected brand+model
    filtered = [
        c for c in all_chunks
        if c.get('brand', '').lower() == brand.lower()
        and c.get('model', '').lower() == model.lower()
    ]

    if not filtered:
        print(f"⚠️  No brochure data found for '{brand} {model}'.")
        print(f"    Available: {list(set(f\"{c['brand']} {c['model']}\" for c in all_chunks[:5]))}")
        return []

    # Optional section filter
    if target_section:
        filtered = [c for c in filtered if c.get('section') == target_section]
        if not filtered:
            print(f"⚠️  No chunks found in section '{target_section}'. Searching all sections.")
            filtered = [c for c in all_chunks if c.get('brand','').lower() == brand.lower()]

    # 2. Embed the user query
    try:
        emb_response = genai.embed_content(
            model='models/gemini-embedding-001',
            content=query
        )
        query_emb = emb_response['embedding']
    except Exception as e:
        print(f"❌ Embedding error: {e}")
        return []

    # 3. Score each chunk using hybrid method
    scored = []
    for chunk in filtered:
        emb = chunk.get('embedding')
        if emb is None:
            continue
        sem = cosine_similarity(query_emb, emb)
        kw  = keyword_score(query, chunk['text'])
        hybrid = 0.8 * sem + 0.2 * kw
        scored.append({
            **chunk,
            'score': hybrid,
            'semantic_score': sem,
            'keyword_score': kw
        })

    # 4. Sort by hybrid score and return top-K
    scored.sort(key=lambda x: x['score'], reverse=True)
    return scored[:limit]


print("✅ RAG Engine ready!")

## Step 5 — Grounded Answer Generator

Sends retrieved chunks to Gemini 2.5 Flash with a strict system prompt to answer **only** from brochure facts — no hallucinations.

In [ ]:
def ask_drivewise(query, brand, model, target_section=None, verbose=True):
    """
    Full DriveWise RAG pipeline:
    1. Retrieve relevant brochure chunks
    2. Generate a grounded, cited answer with Gemini
    3. Return answer + source citations
    """
    start = time.time()

    # ── Retrieval ──────────────────────────────────────────────────────────────
    chunks = retrieve_chunks(query, brand, model, limit=4, target_section=target_section)
    if not chunks:
        return {"answer": f"❌ No brochure data found for '{brand} {model}'.", "sources": [], "response_time": 0}

    # ── Format context for the LLM ─────────────────────────────────────────────
    context_str = ""
    for idx, c in enumerate(chunks):
        context_str += f"\n--- Source [{idx+1}] (Page {c['page']}, Section: {c['section']}) ---\n{c['text']}\n"

    # ── System Prompt (strict grounding rules) ─────────────────────────────────
    system_instruction = f"""You are an expert automotive assistant for the Drive Wise application.
Your job is to answer the user's query about the car: {brand} {model}.

You MUST follow these rules:
1. Ground your answer ONLY in the provided brochure excerpts under "Brochure Context".
2. Do NOT use any pre-existing knowledge. If the information is not in the context, state:
   "I'm sorry, but that information is not available in the brochure details for this vehicle."
3. Include inline citations using [1], [2], etc., matching the Source index in the context.
4. Keep your answer professional, clear, and easy for a car buyer to understand."""

    prompt = f"""Brochure Context for {brand} {model}:
{context_str}

User Query: "{query}"

Grounded Answer (with [1], [2] inline citations):"""

    # ── Generate answer ────────────────────────────────────────────────────────
    try:
        gen_model = genai.GenerativeModel(
            model_name="models/gemini-2.5-flash",
            system_instruction=system_instruction
        )
        response = gen_model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(temperature=0.1)
        )
        answer = response.text.strip()
    except Exception as e:
        return {"answer": f"❌ Generation error: {e}", "sources": [], "response_time": 0}

    elapsed = round(time.time() - start, 2)

    # ── Format sources ─────────────────────────────────────────────────────────
    sources = []
    for idx, c in enumerate(chunks):
        sources.append({
            "citation_num": idx + 1,
            "section": c["section"],
            "page": c["page"],
            "source_file": c["source_file"],
            "score": round(c["score"], 4),
            "semantic_score": round(c["semantic_score"], 4),
            "keyword_score": round(c["keyword_score"], 4),
            "text_preview": c["text"][:150] + "..."
        })

    # ── Pretty Print ───────────────────────────────────────────────────────────
    if verbose:
        print("━" * 70)
        print(f"🚗  {brand} {model}  |  Query: '{query}'")
        print("━" * 70)
        print("\n🤖 Drive Wise Answer:\n")
        print(answer)
        print("\n" + "─" * 70)
        print(f"📚 Retrieved Sources ({len(sources)} chunks) | ⏱️  Response time: {elapsed}s")
        print("─" * 70)
        for s in sources:
            print(f"  [{s['citation_num']}] {s['source_file']}  |  Page {s['page']}  |  Section: {s['section']}")
            print(f"      Relevance: {s['score']:.4f} (semantic: {s['semantic_score']:.4f}, keyword: {s['keyword_score']:.4f})")
            print(f"      Preview: \"{s['text_preview']}\"")
            print()

    return {"answer": answer, "sources": sources, "response_time": elapsed}


print("✅ Generator ready!")

## Step 6 — LLM-as-a-Judge Quality Evaluator

Scores the RAG response on 3 dimensions (1.0–5.0):
- **Faithfulness** — Is the answer grounded in retrieved chunks? No hallucinations?
- **Context Relevance** — Were the right chunks retrieved for the query?
- **Answer Correctness** — Does the answer fully and correctly resolve the query?

In [ ]:
def evaluate_rag_quality(query, context_str, answer):
    """
    Uses Gemini as an LLM-as-a-Judge to score RAG quality on 3 dimensions.
    Returns scores from 1.0 (worst) to 5.0 (best).
    """
    eval_prompt = f"""
You are a RAG quality evaluation judge. Evaluate this RAG system's output.

Query: "{query}"

Retrieved Context:
{context_str}

Generated Answer: "{answer}"

Score each metric from 1.0 (worst) to 5.0 (best):
1. context_relevance: Are the retrieved chunks relevant and helpful for answering the query?
2. faithfulness: Is the answer 100% grounded in the retrieved context without hallucinations?
3. answer_correctness: Does the answer accurately and completely resolve the query?

Respond with ONLY a valid JSON object (no markdown, no explanation):
{{"context_relevance": float, "faithfulness": float, "answer_correctness": float, "rationale": "brief explanation"}}
"""

    try:
        eval_model = genai.GenerativeModel("models/gemini-2.5-flash")
        resp = eval_model.generate_content(eval_prompt)
        text = resp.text.strip()
        # Clean any accidental markdown wrapping
        if text.startswith("```json"): text = text[7:]
        if text.startswith("```"): text = text[3:]
        if text.endswith("```"): text = text[:-3]
        data = json.loads(text.strip())
        return data
    except Exception as e:
        print(f"Evaluation error: {e}")
        return {"context_relevance": 0, "faithfulness": 0, "answer_correctness": 0, "rationale": str(e)}


print("✅ Evaluator ready!")

---
# 🧪 Demo Queries

Run the cells below to test the system with pre-set queries across different cars and topics.

### Demo 1 — Honda Amaze: Safety Features

In [ ]:
result1 = ask_drivewise(
    query="What standard safety features does the Honda Amaze have?",
    brand="Honda",
    model="Amaze"
)

### Demo 2 — Mahindra Thar: Engine & Performance

In [ ]:
result2 = ask_drivewise(
    query="What are the engine options and power output of the Mahindra Thar?",
    brand="Mahindra",
    model="Thar"
)

### Demo 3 — Toyota Innova Hycross: Fuel Efficiency

In [ ]:
result3 = ask_drivewise(
    query="What is the fuel efficiency or mileage of the Toyota Innova Hycross hybrid?",
    brand="Toyota",
    model="Innovahycross"
)

### Demo 4 — Tata Sierra: Infotainment & Tech

In [ ]:
result4 = ask_drivewise(
    query="What are the infotainment and connected car features in the Tata Sierra?",
    brand="Tata",
    model="Sierra"
)

### Demo 5 — Hyundai Verna: Dimensions

In [ ]:
result5 = ask_drivewise(
    query="What are the dimensions and ground clearance of the Hyundai Verna?",
    brand="Hyundai",
    model="Verna"
)

---
## 📊 Quality Evaluation (LLM-as-a-Judge)

Evaluate all 5 demo queries automatically.

In [ ]:
# Evaluate all demo queries
eval_cases = [
    ("What standard safety features does the Honda Amaze have?",             result1),
    ("What are the engine options and power output of the Mahindra Thar?",  result2),
    ("What is the fuel efficiency of the Toyota Innova Hycross hybrid?",    result3),
    ("What are the infotainment features in the Tata Sierra?",              result4),
    ("What are the dimensions and ground clearance of the Hyundai Verna?", result5),
]

eval_results = []
print("Evaluating RAG quality with LLM-as-a-Judge...\n")

for i, (query, result) in enumerate(eval_cases):
    print(f"  Evaluating query {i+1}/5: '{query[:55]}...'")
    if not result["sources"]:
        print("    ⚠️  Skipped (no sources retrieved)")
        continue

    # Reconstruct context string for evaluator
    ctx = "".join([f"[{s['citation_num']}] {s['text_preview']}\n" for s in result["sources"]])
    scores = evaluate_rag_quality(query, ctx, result["answer"])
    scores["query"] = query
    scores["response_time"] = result["response_time"]
    eval_results.append(scores)
    time.sleep(2)  # avoid rate limits

# ── Summary Table ──────────────────────────────────────────────────────────────
print("\n" + "═" * 90)
print("📊 RAG QUALITY EVALUATION RESULTS")
print("═" * 90)
print(f"{'Query':<52} {'Faithfulness':>12} {'Ctx Relevance':>13} {'Correctness':>11} {'Latency':>8}")
print("─" * 90)

avg_faith, avg_ctx, avg_corr = [], [], []
for r in eval_results:
    q = r["query"][:50] + ".." if len(r["query"]) > 50 else r["query"]
    print(f"  {q:<50} {r.get('faithfulness',0):>12.1f} {r.get('context_relevance',0):>13.1f} {r.get('answer_correctness',0):>11.1f} {r.get('response_time',0):>7.2f}s")
    avg_faith.append(r.get("faithfulness", 0))
    avg_ctx.append(r.get("context_relevance", 0))
    avg_corr.append(r.get("answer_correctness", 0))

print("─" * 90)
print(f"  {'AVERAGE':<50} {sum(avg_faith)/len(avg_faith):>12.2f} {sum(avg_ctx)/len(avg_ctx):>13.2f} {sum(avg_corr)/len(avg_corr):>11.2f}")
print("═" * 90)
print("\n📝 Evaluation Rationale (Qualitative Feedback):")
for i, r in enumerate(eval_results):
    print(f"  [{i+1}] {r.get('rationale', 'N/A')}")

---
## 🎮 Interactive Mode — Ask Your Own Question!

Change the values below and run the cell to test any query.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║            ✏️  CHANGE THESE VALUES TO TEST             ║
# ╚══════════════════════════════════════════════════════════╝

MY_QUERY  = "What is the boot space and seating capacity?"  # ← Your question
MY_BRAND  = "Mahindra"                                       # ← Brand name
MY_MODEL  = "Xuv700"                                         # ← Model name

# Available cars:
# Honda / Amaze
# Hyundai / Aura, Grand I10 Nios, Verna  
# Mahindra / Scorpio N, Thar, Xuv700
# Tata / Sierra
# Toyota / Fortuner, Innovahycross

# ── Run Query ─────────────────────────────────────────────
my_result = ask_drivewise(MY_QUERY, MY_BRAND, MY_MODEL)

In [ ]:
# Optional: evaluate the quality of your query
if my_result["sources"]:
    ctx = "".join([f"[{s['citation_num']}] {s['text_preview']}\n" for s in my_result["sources"]])
    my_scores = evaluate_rag_quality(MY_QUERY, ctx, my_result["answer"])
    print("\n📊 Quality Evaluation:")
    print(f"  Faithfulness:      {my_scores.get('faithfulness', 0):.1f} / 5.0")
    print(f"  Context Relevance: {my_scores.get('context_relevance', 0):.1f} / 5.0")
    print(f"  Answer Correctness:{my_scores.get('answer_correctness', 0):.1f} / 5.0")
    print(f"  Rationale: {my_scores.get('rationale', '')}")

---
## 🔍 Explore the Index

Inspect the raw index data — what cars are indexed, chunk counts, and a sample chunk.

In [ ]:
# Show all available cars with details
print("📁 Full Index Summary")
print("═" * 60)
all_chunks = index_data.get('chunks', [])

# Build a map of brand → models → sections
car_map = {}
for c in all_chunks:
    key = (c.get('brand', '?'), c.get('model', '?'))
    if key not in car_map:
        car_map[key] = {"count": 0, "sections": set()}
    car_map[key]["count"] += 1
    car_map[key]["sections"].add(c.get('section', 'Unknown'))

for (brand, model), info in sorted(car_map.items()):
    print(f"\n  🚗 {brand} {model}")
    print(f"     Chunks: {info['count']}")
    print(f"     Sections covered:")
    for sec in sorted(info["sections"]):
        print(f"       • {sec}")

print(f"\n{'═'*60}")
print(f"Total: {len(all_chunks)} chunks across {len(car_map)} cars")

In [ ]:
# Show a sample chunk to understand the data structure
sample = all_chunks[0].copy()
emb_preview = list(sample.get('embedding', [])[:5]) if sample.get('embedding') is not None else []
sample['embedding'] = f"[{emb_preview[0]:.4f}, {emb_preview[1]:.4f}, ... ({len(all_chunks[0].get('embedding', []))} dims)]"

print("📄 Sample Chunk Structure:")
print("─" * 60)
for k, v in sample.items():
    if k == 'text':
        print(f"  text      : '{str(v)[:120]}...'")
    else:
        print(f"  {k:<10}: {v}")

---
<div style="background: #f0fdf4; border: 1px solid #86efac; border-radius: 12px; padding: 1.5rem; text-align: center;">
  <h3 style="color: #166534; margin: 0;">✅ DriveWise Demo Complete!</h3>
  <p style="color: #15803d; margin: 0.5rem 0 0 0;">
    This notebook demonstrates the full RAG pipeline: PDF indexing → embedding → hybrid retrieval → grounded generation → LLM evaluation
  </p>
</div>

### 📌 Key Design Decisions

| Component | Choice | Reason |
|-----------|--------|---------|
| Embeddings | `gemini-embedding-001` | High-quality multilingual embeddings |
| Generation | `gemini-2.5-flash` | Fast, cost-efficient, strong instruction following |
| Retrieval | Hybrid (semantic + keyword) | Captures both meaning AND exact spec numbers |
| Chunking | Paragraph-level (~800 chars) | Balances context and precision |
| Section Tags | Rule-based + keyword scoring | Metadata pre-filter reduces noise |
| Evaluation | LLM-as-a-Judge | Scalable quality measurement without ground truth |